In [1]:
import pandas as pd
import os
import json
from pathlib import Path

In [2]:
df_dev = pd.read_json("/Users/afazl/Documents/GitHub/interpretability-llms-agents/implementations/agentic_vqa_eval/FinQA_dataset/dev.json")
# df_test = pd.read_json("data/FinQA/FinQA_test.jsonl")

In [3]:
df_dev.head()

,pre_text,post_text,filename,table_ori,table,qa,id,table_retrieved,text_retrieved,table_retrieved_all,text_retrieved_all
0,[largest operators of open-loop and closed-loo...,"[( 1 ) visa inc ., figures as reported previou...",V/2008/page_17.pdf,"[[Company, Payments Volume (billions), Total ...","[[company, payments volume ( billions ), total...",{'question': 'what is the average payment volu...,V/2008/page_17.pdf-1,"[{'score': 3.21082329750061, 'ind': 'table_3'}...","[{'score': -1.715970516204834, 'ind': 'text_43...","[{'score': 3.21082329750061, 'ind': 'table_3'}...","[{'score': -1.715970516204834, 'ind': 'text_43..."
1,[performance graph comparison of five-year cum...,[.],C/2017/page_328.pdf,"[[DATE, CITI, S&P 500, S&P FINANCIALS], [31-De...","[[date, citi, s&p 500, s&p financials], [31-de...",{'question': 'what was the percentage cumulati...,C/2017/page_328.pdf-1,"[{'score': 2.015230894088745, 'ind': 'table_6'...","[{'score': 0.8624499440193171, 'ind': 'text_1'}]","[{'score': 2.015230894088745, 'ind': 'table_6'...","[{'score': 0.8624499440193171, 'ind': 'text_1'..."
2,[the acquisition date is on or after the begin...,[.],DVN/2007/page_58.pdf,"[[, Oil (MMBbls), Gas (Bcf), NGLs (MMBbl...","[[, oil ( mmbbls ), gas ( bcf ), ngls ( mmbbls...",{'question': 'what percentage of the total oil...,DVN/2007/page_58.pdf-2,"[{'score': 2.8360869884490962, 'ind': 'table_3...",[],"[{'score': 2.8360869884490962, 'ind': 'table_3...","[{'score': 0.065325163304805, 'ind': 'text_21'..."
3,"[entergy mississippi , inc ., management 2019s...",[the volume/weather variance is primarily due ...,ETR/2011/page_341.pdf,"[[, Amount (In Millions)], [2009 net revenue, ...","[[, amount ( in millions )], [2009 net revenue...",{'question': 'in 2010 what was the net change ...,ETR/2011/page_341.pdf-3,"[{'score': 2.869751691818237, 'ind': 'table_4'...","[{'score': 0.28362259268760603, 'ind': 'text_3'}]","[{'score': 2.869751691818237, 'ind': 'table_4'...","[{'score': 0.28362259268760603, 'ind': 'text_3..."
4,"[entergy louisiana , inc ., management's finan...",[the deferred fuel cost revisions variance res...,ETR/2004/page_213.pdf,"[[, (In Millions)], [2002 net revenue, $922.9]...","[[, ( in millions )], [2002 net revenue, $ 922...",{'question': 'what are the deferred fuel cost ...,ETR/2004/page_213.pdf-2,"[{'score': 2.318804502487182, 'ind': 'table_2'}]","[{'score': 0.695305705070495, 'ind': 'text_6'}...","[{'score': 2.318804502487182, 'ind': 'table_2'...","[{'score': 0.695305705070495, 'ind': 'text_6'}..."


In [4]:
print("Columns in dev dataset: {}".format(df_dev.columns.tolist()))
print("Number of rows in dev dataset: {}".format(len(df_dev)))

Columns in dev dataset: ['pre_text', 'post_text', 'filename', 'table_ori', 'table', 'qa', 'id', 'table_retrieved', 'text_retrieved', 'table_retrieved_all', 'text_retrieved_all']
Number of rows in dev dataset: 883


In [5]:
df_dev['question'] = df_dev['qa'].apply(lambda x: x['question'])

In [6]:
df_dev

,pre_text,post_text,filename,table_ori,table,qa,id,table_retrieved,text_retrieved,table_retrieved_all,text_retrieved_all,question
0,[largest operators of open-loop and closed-loo...,"[( 1 ) visa inc ., figures as reported previou...",V/2008/page_17.pdf,"[[Company, Payments Volume (billions), Total ...","[[company, payments volume ( billions ), total...",{'question': 'what is the average payment volu...,V/2008/page_17.pdf-1,"[{'score': 3.21082329750061, 'ind': 'table_3'}...","[{'score': -1.715970516204834, 'ind': 'text_43...","[{'score': 3.21082329750061, 'ind': 'table_3'}...","[{'score': -1.715970516204834, 'ind': 'text_43...",what is the average payment volume per transac...
1,[performance graph comparison of five-year cum...,[.],C/2017/page_328.pdf,"[[DATE, CITI, S&P 500, S&P FINANCIALS], [31-De...","[[date, citi, s&p 500, s&p financials], [31-de...",{'question': 'what was the percentage cumulati...,C/2017/page_328.pdf-1,"[{'score': 2.015230894088745, 'ind': 'table_6'...","[{'score': 0.8624499440193171, 'ind': 'text_1'}]","[{'score': 2.015230894088745, 'ind': 'table_6'...","[{'score': 0.8624499440193171, 'ind': 'text_1'...",what was the percentage cumulative total retur...
2,[the acquisition date is on or after the begin...,[.],DVN/2007/page_58.pdf,"[[, Oil (MMBbls), Gas (Bcf), NGLs (MMBbl...","[[, oil ( mmbbls ), gas ( bcf ), ngls ( mmbbls...",{'question': 'what percentage of the total oil...,DVN/2007/page_58.pdf-2,"[{'score': 2.8360869884490962, 'ind': 'table_3...",[],"[{'score': 2.8360869884490962, 'ind': 'table_3...","[{'score': 0.065325163304805, 'ind': 'text_21'...",what percentage of the total oil and gas mmboe...
3,"[entergy mississippi , inc ., management 2019s...",[the volume/weather variance is primarily due ...,ETR/2011/page_341.pdf,"[[, Amount (In Millions)], [2009 net revenue, ...","[[, amount ( in millions )], [2009 net revenue...",{'question': 'in 2010 what was the net change ...,ETR/2011/page_341.pdf-3,"[{'score': 2.869751691818237, 'ind': 'table_4'...","[{'score': 0.28362259268760603, 'ind': 'text_3'}]","[{'score': 2.869751691818237, 'ind': 'table_4'...","[{'score': 0.28362259268760603, 'ind': 'text_3...",in 2010 what was the net change in net revenue...
4,"[entergy louisiana , inc ., management's finan...",[the deferred fuel cost revisions variance res...,ETR/2004/page_213.pdf,"[[, (In Millions)], [2002 net revenue, $922.9]...","[[, ( in millions )], [2002 net revenue, $ 922...",{'question': 'what are the deferred fuel cost ...,ETR/2004/page_213.pdf-2,"[{'score': 2.318804502487182, 'ind': 'table_2'}]","[{'score': 0.695305705070495, 'ind': 'text_6'}...","[{'score': 2.318804502487182, 'ind': 'table_2'...","[{'score': 0.695305705070495, 'ind': 'text_6'}...",what are the deferred fuel cost revisions as a...
...,...,...,...,...,...,...,...,...,...,...,...,...
878,[contractual obligations and commercial commit...,"[we have no long-term debt , capital leases or...",ABMD/2007/page_52.pdf,"[[, Payments Due By Fiscal Year], [Contractual...","[[contractual obligations, payments due by fis...",{'question': 'what portion of total future obl...,ABMD/2007/page_52.pdf-4,"[{'score': 2.291633367538452, 'ind': 'table_2'...","[{'score': -0.171677857637405, 'ind': 'text_0'...","[{'score': 2.291633367538452, 'ind': 'table_2'...","[{'score': -0.171677857637405, 'ind': 'text_0'...",what portion of total future obligations is re...
879,"[abiomed , inc ., and subsidiaries notes to co...","[from time-to-time , the company is involved i...",ABMD/2006/page_75.pdf,"[[Fiscal Year Ending March 31,, Operating Leas...","[[fiscal year ending march 31,, operating leas...",{'question': 'what percentage of total future ...,ABMD/2006/page_75.pdf-4,"[{'score': 3.150706529617309, 'ind': 'table_5'...",[],"[{'score': 3.150706529617309, 'ind': 'table_5'...","[{'score': -0.760037362575531, 'ind': 'text_21...",what percentage of total future minimum lease ...
880,[contingent consideration of up to $ 13.8 mill...,[the fair values of these investments a

In [7]:
df_dev_updated = df_dev[['pre_text', 'post_text', 'table_ori', 'table', 'filename', 'id', 'question']]

In [8]:
df_dev_updated

,pre_text,post_text,table_ori,table,filename,id,question
0,[largest operators of open-loop and closed-loo...,"[( 1 ) visa inc ., figures as reported previou...","[[Company, Payments Volume (billions), Total ...","[[company, payments volume ( billions ), total...",V/2008/page_17.pdf,V/2008/page_17.pdf-1,what is the average payment volume per transac...
1,[performance graph comparison of five-year cum...,[.],"[[DATE, CITI, S&P 500, S&P FINANCIALS], [31-De...","[[date, citi, s&p 500, s&p financials], [31-de...",C/2017/page_328.pdf,C/2017/page_328.pdf-1,what was the percentage cumulative total retur...
2,[the acquisition date is on or after the begin...,[.],"[[, Oil (MMBbls), Gas (Bcf), NGLs (MMBbl...","[[, oil ( mmbbls ), gas ( bcf ), ngls ( mmbbls...",DVN/2007/page_58.pdf,DVN/2007/page_58.pdf-2,what percentage of the total oil and gas mmboe...
3,"[entergy mississippi , inc ., management 2019s...",[the volume/weather variance is primarily due ...,"[[, Amount (In Millions)], [2009 net revenue, ...","[[, amount ( in millions )], [2009 net revenue...",ETR/2011/page_341.pdf,ETR/2011/page_341.pdf-3,in 2010 what was the net change in net revenue...
4,"[entergy louisiana , inc ., management's finan...",[the deferred fuel cost revisions variance res...,"[[, (In Millions)], [2002 net revenue, $922.9]...","[[, ( in millions )], [2002 net revenue, $ 922...",ETR/2004/page_213.pdf,ETR/2004/page_213.pdf-2,what are the deferred fuel cost revisions as a...
...,...,...,...,...,...,...,...
878,[contractual obligations and commercial commit...,"[we have no long-term debt , capital leases or...","[[, Payments Due By Fiscal Year], [Contractual...","[[contractual obligations, payments due by fis...",ABMD/2007/page_52.pdf,ABMD/2007/page_52.pdf-4,what portion of total future obligations is re...
879,"[abiomed , inc ., and subsidiaries notes to co...","[from time-to-time , the company is involved i...","[[Fiscal Year Ending March 31,, Operating Leas...","[[fiscal year ending march 31,, operating leas...",ABMD/2006/page_75.pdf,ABMD/2006/page_75.pdf-4,what percentage of total future minimum lease ...
880,[contingent consideration of up to $ 13.8 mill...,[the fair values of these investments are base...,"[[, 2011, 2010], [Money market funds, $17,187,...","[[, 2011, 2010], [money market funds, $ 17187,...",ADI/2011/page_81.pdf,ADI/2011/page_81.pdf-3,what was the percentage increase of total defe...
881,[we hold an interest rate swap agreement to he...,[fair value of forward exchange contracts afte...,"[[, October 29, 2011, October 30, 2010], [Fair...","[[, october 29 2011, october 30 2010], [fair v...",ADI/2011/page_50.pdf,ADI/2011/page_50.pdf-3,what would the amount accrued because of inter...


In [9]:
df_dev_updated[df_dev_updated['filename']!=df_dev_updated['id']]

,pre_text,post_text,table_ori,table,filename,id,question
0,[largest operators of open-loop and closed-loo...,"[( 1 ) visa inc ., figures as reported previou...","[[Company, Payments Volume (billions), Total ...","[[company, payments volume ( billions ), total...",V/2008/page_17.pdf,V/2008/page_17.pdf-1,what is the average payment volume per transac...
1,[performance graph comparison of five-year cum...,[.],"[[DATE, CITI, S&P 500, S&P FINANCIALS], [31-De...","[[date, citi, s&p 500, s&p financials], [31-de...",C/2017/page_328.pdf,C/2017/page_328.pdf-1,what was the percentage cumulative total retur...
2,[the acquisition date is on or after the begin...,[.],"[[, Oil (MMBbls), Gas (Bcf), NGLs (MMBbl...","[[, oil ( mmbbls ), gas ( bcf ), ngls ( mmbbls...",DVN/2007/page_58.pdf,DVN/2007/page_58.pdf-2,what percentage of the total oil and gas mmboe...
3,"[entergy mississippi , inc ., management 2019s...",[the volume/weather variance is primarily due ...,"[[, Amount (In Millions)], [2009 net revenue, ...","[[, amount ( in millions )], [2009 net revenue...",ETR/2011/page_341.pdf,ETR/2011/page_341.pdf-3,in 2010 what was the net change in net revenue...
4,"[entergy louisiana , inc ., management's finan...",[the deferred fuel cost revisions variance res...,"[[, (In Millions)], [2002 net revenue, $922.9]...","[[, ( in millions )], [2002 net revenue, $ 922...",ETR/2004/page_213.pdf,ETR/2004/page_213.pdf-2,what are the deferred fuel cost revisions as a...
...,...,...,...,...,...,...,...
878,[contractual obligations and commercial commit...,"[we have no long-term debt , capital leases or...","[[, Payments Due By Fiscal Year], [Contractual...","[[contractual obligations, payments due by fis...",ABMD/2007/page_52.pdf,ABMD/2007/page_52.pdf-4,what portion of total future obligations is re...
879,"[abiomed , inc ., and subsidiaries notes to co...","[from time-to-time , the company is involved i...","[[Fiscal Year Ending March 31,, Operating Leas...","[[fiscal year ending march 31,, operating leas...",ABMD/2006/page_75.pdf,ABMD/2006/page_75.pdf-4,what percentage of total future minimum lease ...
880,[contingent consideration of up to $ 13.8 mill...,[the fair values of these investments are base...,"[[, 2011, 2010], [Money market funds, $17,187,...","[[, 2011, 2010], [money market funds, $ 17187,...",ADI/2011/page_81.pdf,ADI/2011/page_81.pdf-3,what was the percentage increase of total defe...
881,[we hold an interest rate swap agreement to he...,[fair value of forward exchange contracts afte...,"[[, October 29, 2011, October 30, 2010], [Fair...","[[, october 29 2011, october 30 2010], [fair v...",ADI/2011/page_50.pdf,ADI/2011/page_50.pdf-3,what would the amount accrued because of inter...


In [12]:
df_dev_updated.to_json("/Users/afazl/Documents/GitHub/interpretability-llms-agents/implementations/agentic_vqa_eval/FinQA_dataset/dev_updated.json", orient="records", lines=True)

In [19]:
df_test = pd.read_json("/Users/afazl/Documents/GitHub/interpretability-llms-agents/implementations/agentic_vqa_eval/FinQA_dataset/dev_updated.json", lines=True)

In [20]:
df_test

,pre_text,post_text,table_ori,table,filename,id,question
0,[largest operators of open-loop and closed-loo...,"[( 1 ) visa inc ., figures as reported previou...","[[Company, Payments Volume (billions), Total ...","[[company, payments volume ( billions ), total...",V/2008/page_17.pdf,V/2008/page_17.pdf-1,what is the average payment volume per transac...
1,[performance graph comparison of five-year cum...,[.],"[[DATE, CITI, S&P 500, S&P FINANCIALS], [31-De...","[[date, citi, s&p 500, s&p financials], [31-de...",C/2017/page_328.pdf,C/2017/page_328.pdf-1,what was the percentage cumulative total retur...
2,[the acquisition date is on or after the begin...,[.],"[[, Oil (MMBbls), Gas (Bcf), NGLs (MMBbl...","[[, oil ( mmbbls ), gas ( bcf ), ngls ( mmbbls...",DVN/2007/page_58.pdf,DVN/2007/page_58.pdf-2,what percentage of the total oil and gas mmboe...
3,"[entergy mississippi , inc ., management 2019s...",[the volume/weather variance is primarily due ...,"[[, Amount (In Millions)], [2009 net revenue, ...","[[, amount ( in millions )], [2009 net revenue...",ETR/2011/page_341.pdf,ETR/2011/page_341.pdf-3,in 2010 what was the net change in net revenue...
4,"[entergy louisiana , inc ., management's finan...",[the deferred fuel cost revisions variance res...,"[[, (In Millions)], [2002 net revenue, $922.9]...","[[, ( in millions )], [2002 net revenue, $ 922...",ETR/2004/page_213.pdf,ETR/2004/page_213.pdf-2,what are the deferred fuel cost revisions as a...
...,...,...,...,...,...,...,...
878,[contractual obligations and commercial commit...,"[we have no long-term debt , capital leases or...","[[, Payments Due By Fiscal Year], [Contractual...","[[contractual obligations, payments due by fis...",ABMD/2007/page_52.pdf,ABMD/2007/page_52.pdf-4,what portion of total future obligations is re...
879,"[abiomed , inc ., and subsidiaries notes to co...","[from time-to-time , the company is involved i...","[[Fiscal Year Ending March 31,, Operating Leas...","[[fiscal year ending march 31,, operating leas...",ABMD/2006/page_75.pdf,ABMD/2006/page_75.pdf-4,what percentage of total future minimum lease ...
880,[contingent consideration of up to $ 13.8 mill...,[the fair values of these investments are base...,"[[, 2011, 2010], [Money market funds, $17,187,...","[[, 2011, 2010], [money market funds, $ 17187,...",ADI/2011/page_81.pdf,ADI/2011/page_81.pdf-3,what was the percentage increase of total defe...
881,[we hold an interest rate swap agreement to he...,[fair value of forward exchange contracts afte...,"[[, October 29, 2011, October 30, 2010], [Fair...","[[, october 29 2011, october 30 2010], [fair v...",ADI/2011/page_50.pdf,ADI/2011/page_50.pdf-3,what would the amount accrued because of inter...
